# 11 — H1's damage axis, registered on the corrected instrument, on prompts nothing has touched

**The state this exists to fix.** H1 is the project's original hypothesis and it is currently in the
least defensible position of any claim here:

- Correction 22 registered a damage-axis test, ran it on 48 unseen prompts, and recorded
  **NOT SUPPORTED**.
- Correction 23 then showed the instrument that verdict was computed with disagrees with blind hand
  labels 31% of the time on degraded text — and re-scoring the *same* generations under the corrected
  thresholds reverses it: `absproj-0.90` at 66.7% broken against dense's 100.0% and static's 95.8%,
  disjoint against both.

A null overturned by re-scoring after the fact is not a result in either direction. Changing an
instrument after seeing a null and keeping the new number is the single most suspect move available,
and this notebook exists so that nobody has to trust it. The instrument is now **frozen and
published** — `data/results/week6_labels.json`, fitted on 108 labels that are themselves
human-confirmed at 96.4% on this exact axis (correction 26) — the rule is written below before the
run, and the prompts are 48 that no run, mask, threshold or selection step in this project has ever
seen.

§1.2 **hard-stops** if the corrected thresholds are missing. Three runs of notebook 10 silently used
the wrong ruler; H1's verdict is precisely the quantity that depends on which ruler is used, so here
the fallback is not allowed to happen quietly.

About fifteen minutes on a GPU.

## §0 Setup — identical to notebooks 07, 08 and 10

In [ ]:
# %% 0.0 BOOTSTRAP -- run this first, always. Identical locally and on Colab.
import os, subprocess, sys
from pathlib import Path

GITHUB_REPO = "YarinShitrit/adass"                 # from `git remote -v`
DRIVE_DIR   = "/content/drive/MyDrive/adass"       # fallback if you skip GitHub

IN_COLAB = "google.colab" in sys.modules


def _find_root(start):
    """Walk up looking for the repo: a pyproject.toml sitting next to the adass package."""
    p = Path(start).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").is_file() and (cand / "adass" / "core.py").is_file():
            return cand
    return None


def _early_env():
    """Read a .env BEFORE the package exists. Mirrors adass.env, which is the canonical copy.

    Duplicated here because on Colab this cell runs before the repo is cloned and before anything
    is pip-installed -- and the GitHub token needed to perform the clone has to come from
    somewhere. Which is also why the repo's own .env cannot be that somewhere: .env is gitignored,
    so a clone never contains one. Keep a filled-in .env on Drive; it survives runtimes.
    """
    for p in (Path.cwd() / ".env", Path("/content/drive/MyDrive/adass/.env"),
              Path("/content/drive/MyDrive/.env"), Path("/content/.env"), Path.home() / ".env"):
        if p.is_file():
            for line in p.read_text(encoding="utf-8").splitlines():
                line = line.strip().removeprefix("export ")
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, _, v = line.partition("=")
                v = v.strip().strip("\"'")
                if v and not os.environ.get(k.strip()):
                    os.environ[k.strip()] = v
            print(f"loaded .env from {p}")
            return p
    return None


def _secret(name, prompt):
    """Environment (incl. .env) -> Colab Secrets -> prompt. Nothing is stored in the notebook."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata          # browser Colab frontend only
        v = userdata.get(name)
        if v:
            os.environ[name] = v
            return v
    except Exception:
        pass
    import getpass
    v = getpass.getpass(prompt)
    if v:
        os.environ[name] = v
    return v


def _clone_or_update(repo, token, dest):
    """Clone if absent, pull if already there. NEVER let the token reach a traceback.

    Two failures this exists for, both hit on 30 August. A kernel restart leaves /content intact,
    so `git clone` into an existing checkout dies with exit 128 and a message nobody sees; and
    `check=True` raises CalledProcessError, whose `.args` carries the tokenised URL straight into
    the traceback Colab prints and then saves into the notebook file. A leaked PAT is a worse
    outcome than a failed clone, so the token never travels with the exception.
    """
    dest = Path(dest)
    if (dest / ".git").is_dir():
        cmd, what = ["git", "-C", str(dest), "pull", "--quiet"], "pull"
    elif dest.exists() and any(dest.iterdir()):
        raise RuntimeError(f"{dest} exists and is not a git checkout. Remove it, or point ROOT at "
                           "the repo by hand.")
    else:
        cmd, what = ["git", "clone", "--quiet",
                     f"https://{token}@github.com/{repo}.git", str(dest)], "clone"
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        err = (r.stderr or "").strip()
        if token:
            err = err.replace(token, "<token>")
        raise RuntimeError(f"git {what} failed (exit {r.returncode}): {err[:400]}")
    print(f"git {what} ok -> {dest}")
    return dest


_early_env()
ROOT = _find_root(Path.cwd())

if IN_COLAB and ROOT is None:
    if not os.environ.get("GH_TOKEN") and Path("/content/drive").exists() is False:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            _early_env()
        except Exception:
            pass
    token = _secret("GH_TOKEN", "GitHub PAT (read access to the repo): ")
    if token:
        ROOT = _clone_or_update(GITHUB_REPO, token, "/content/adass")
    else:
        ROOT = _find_root(DRIVE_DIR) or Path(DRIVE_DIR)

assert ROOT is not None, "repo not found -- set GITHUB_REPO, or put the repo at DRIVE_DIR"
os.chdir(ROOT)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
elif str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


if IN_COLAB:
    # STALENESS CHECK. _find_root short-circuits the clone/pull above whenever a checkout
    # already exists, so a runtime cloned before a push keeps running old code and old data
    # files forever, and re-running the notebook cannot fix it. That cost three runs of
    # notebook 10 on 30-31 August: data/results/week6_labels.json was on the remote and simply
    # not in the runtime, so the corrected thresholds were never found and every verdict came
    # out of the fallback. Report rather than pull: a run in progress has already modified
    # tracked files under data/results/, and a pull that fails halfway is worse than none.
    try:
        subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], timeout=90)
        _behind = subprocess.run(["git", "-C", str(ROOT), "rev-list", "--count",
                                  "HEAD..origin/main"],
                                 capture_output=True, text=True).stdout.strip()
        if _behind and _behind != "0":
            print("!" * 78)
            print(f"!! THIS CHECKOUT IS {_behind} COMMIT(S) BEHIND origin/main.")
            print("!! Anything added since then is missing here, silently. Either start a fresh")
            print("!! runtime, or fetch the specific file you need, e.g.")
            print(f"!!   !cd {ROOT} && git checkout origin/main -- data/results/<file>.json")
            print("!" * 78)
        else:
            print("checkout is up to date with origin/main")
    except Exception as _e:
        print("staleness check skipped:", _e)

import adass
adass.load_env()
# Both sources are GATED: HF_TOKEN needs google/gemma-2-2b-it AND walledai/AdvBench accepted.
# NOTE: HF_HUB_OFFLINE is deliberately NOT set -- nothing is cached on a fresh runtime.
from huggingface_hub import get_token
if not get_token():
    adass.require("HF_TOKEN", "HuggingFace token (gemma-2-2b-it + AdvBench accepted): ")

import torch
print(adass.paths.describe())
print(adass.env.status())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none -- everything past §1 will be very slow")

In [ ]:
# %% 0.1 Run flags, the truncation point, and the prior run.
import json, math, itertools
from collections import Counter

OUT = "week8_h1_registered.json"   # adass.save_results resolves bare names to data/results/

LOAD_MODEL = os.environ.get("ADASS_LOAD_MODEL", "0") == "1"
print(f"LOAD_MODEL={LOAD_MODEL}   (set ADASS_LOAD_MODEL=1 for everything past §1)")
RESULTS = {}

# save_results MERGES at the top level, so no later cell can delete a section it did not compute.
# §0.1 is the one deliberate truncation point, and it fires only on a full run: a CPU-only pass
# cannot regenerate the GPU sections, so rotating them away would destroy the only copy. That is
# not hypothetical -- it happened on 21 August, and again in a milder form on 25 August, which is
# why the `and LOAD_MODEL` is there.
_out = adass.results_path(OUT)
_prev = _out.with_suffix(".prev.json")
if _out.exists() and LOAD_MODEL:
    _out.replace(_prev)
    print(f"rotated {_out.name} -> {_prev.name}  (full run: starting clean)")
elif _out.exists():
    print(f"CPU-only run: MERGING into existing {_out.name}, not rotating.")

# The verdict cells re-run on CPU against whatever the last GPU run left behind, so every one of
# them reads through `stored()` rather than off a local variable that only exists mid-run.
# WHERE THE RESULTS SURVIVE. A Colab runtime takes its filesystem with it when it is recycled,
# and on 27 August it did exactly that to a completed run: every section had saved, every save had
# gone to /content/adass, and /content/adass no longer existed. Only the notebook's printed cell
# outputs were left -- the rates, but none of the generations or per-item instrument arrays.
#
# adass.save_results mirrors every write to ADASS_MIRROR when that is set. Point it at Drive and
# the copy outlives the runtime. This block sets it up automatically on Colab and says so loudly
# when it cannot, because a run that is not mirrored is a run you may have to pay for twice.
if IN_COLAB and not os.environ.get("ADASS_MIRROR"):
    _drive = Path("/content/drive/MyDrive")
    if not _drive.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as _e:
            print(f"could not mount Drive ({_e})")
    if _drive.exists():
        os.environ["ADASS_MIRROR"] = str(_drive / "adass_results")
        print(f"mirroring every save to {os.environ['ADASS_MIRROR']}")
    else:
        print("!! NO MIRROR: results live only on this runtime and die with it.")
        print("!! Set ADASS_MIRROR to a Drive path, or copy data/results/ out before disconnecting.")
elif os.environ.get("ADASS_MIRROR"):
    print(f"mirroring every save to {os.environ['ADASS_MIRROR']}")

PRIOR = {}
for _p in (_out, _prev):
    if _p.exists():
        PRIOR = json.load(open(_p))
        print(f"prior run loaded from {_p.name}: {list(PRIOR)}")
        break


def stored(key, default=None):
    """This run's value if this run computed it, else the previous run's."""
    return RESULTS.get(key, PRIOR.get(key, default))

In [ ]:
# %% 0.2 Module provenance. The stored judge output is only reusable if the prompts still hash
# to the version it was produced under -- see HANDOVER trap 2 for what silent drift cost here.
print("adass      ", adass.__version__, "from", Path(adass.__file__).parent)
print("judge hash ", adass.judge_prompt_hash())
_stored_hash = json.load(open(adass.artifact("steps123_results.json")))["step3"]["prompt_hash"]
print("stored hash", _stored_hash,
      "-- MATCH" if adass.judge_prompt_hash() == _stored_hash else "-- CHANGED: do not reload")
assert adass.judge_prompt_hash() == _stored_hash, (
    "judge prompts changed: every comparison in this notebook against a stored week-4 number "
    "would be measuring two different instruments. Bump the version deliberately or revert.")

In [ ]:
# %% 0.3 Environment, splits, vectors. float16 is a STOP condition: Gemma-2 emits broken text in
# fp16, and that failure is visually identical to the degeneration this project studies.
import torch, transformers

DEV, DT = adass.pick_device(), adass.pick_dtype(adass.pick_device())
print("device", DEV, "| dtype", DT)
assert DT is not torch.float16, "float16: STOP. See README, environment check."

CONFIG = json.load(open(adass.paths.config()))
LAYER  = CONFIG["best_layer"]              # 10, set from evidence on 24 August
R16    = CONFIG["r16"]                     # 0.5537 -- relative strength of the raw vector at L16
assert LAYER == 10, f"config says layer {LAYER}; this notebook is written for the layer-10 point"

SPL     = adass.make_splits(seed=CONFIG["seed"])   # train_n MUST stay at its default 128 --
PROMPTS = SPL["harmless_test"]                     # train_n=160 shifts harmless_test by 32 items
assert len(PROMPTS) == 48
DIRS = torch.load(adass.artifact("refusal_dirs.pt"))
V    = DIRS[LAYER + 1]                             # refusal_dirs is indexed [layer + 1]
print(f"{len(PROMPTS)} test prompts | layer {LAYER} | ||V|| {float(V.norm()):.3f} "
      f"| r16 {R16:.4f}")

MAXNEW  = 128          # 48 tokens cannot show apology-then-answer; week 3 §2 is why this is 128
GEN_BS  = 8
KL_BS   = 4            # teacher-forced logits are [B, T, 256k]; 4 keeps a T4 inside its memory
KL_REF_TOKENS = 48     # the fixed reference text, as in week 3 §5.1
KL_WINDOW = 8          # see §5: the shared window that makes position schemes comparable

RESULTS["env"] = dict(load_model=LOAD_MODEL, device=DEV, dtype=str(DT), torch=torch.__version__,
                      transformers=transformers.__version__, layer=LAYER, r16=R16,
                      max_new_tokens=MAXNEW, n_prompts=len(PROMPTS),
                      gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
                      bf16_supported=(torch.cuda.is_bf16_supported()
                                      if torch.cuda.is_available() else None))
print("env:", RESULTS["env"])
print(adass.save_results(RESULTS, OUT))

MODEL = TOK = TO_CHAT = None
if LOAD_MODEL:
    MODEL, TOK, DT, DEV = adass.load_model()
    TO_CHAT = adass.make_chat_fn(TOK)
    print("model loaded")

## §1 Instruments, the frozen thresholds, and the gate

The gate compares under `FIT_ANCHOR`, the calibration its stored reference was produced with, so it
measures whether the model and pipeline reproduced rather than whether the ruler changed. The arms in
§2 are scored under the corrected thresholds, which is the whole point.

In [ ]:
# %% 1.1 Fit the mechanical thresholds on the anchors, and define the two axes.
GENS = json.load(open(adass.artifact("week3_generations.json")))
FIT = adass.fit_coherence_thresholds(GENS["no-steer"], GENS["dense/all m=2"])
for feat, d in FIT.items():
    print(f"  {feat:12} thr={d['threshold']:8.3f}  bacc={d['balanced_acc']:.3f}  margin={d['margin']:+.3f}")


def mech_broken(texts):
    return [adass.classify_mechanical(t, FIT)["broken"] for t in texts]


def judge_answered(prompts, texts):
    """None when the model is not loaded, so every section still completes."""
    if not LOAD_MODEL:
        return None
    return [o["answered"] for o in
            adass.local_judge_binary(MODEL, TOK, TO_CHAT, list(zip(prompts, texts)))]


def score(texts, prompts, answered=None):
    """The row every table in this notebook reports. Wilson CIs on all three rates.

    CIs are stored as [lo, hi] -- wilson_ci returns (point, lo, hi) and the point estimate is
    already the neighbouring field. Indexing rather than unpacking that tuple is what cost a
    blocking control its meaning on 23 August (WORKLOG correction 14), so the slice is explicit.
    """
    br = mech_broken(texts)
    ans = judge_answered(prompts, texts) if answered is None else answered
    n = len(texts)
    row = dict(n=n, broken=sum(br) / n, broken_ci=list(adass.wilson_ci(sum(br), n))[1:],
               matcher=adass.refusal_rate(texts))
    if ans is not None:
        clean = [(not b) and (not a) for b, a in zip(br, ans)]
        row.update(suppressed=1 - sum(ans) / n,
                   suppressed_ci=list(adass.wilson_ci(n - sum(ans), n))[1:],
                   clean_refusal=sum(clean) / n,
                   clean_refusal_ci=list(adass.wilson_ci(sum(clean), n))[1:],
                   judge_answered=ans)
    row["mech_broken"] = br
    return row


def fmt(row, label=""):
    s = f"{label:30} broken {row['broken']:6.1%}"
    if "clean_refusal" in row:
        s += f" | suppressed {row['suppressed']:6.1%} | CLEAN {row['clean_refusal']:6.1%}"
    if "kl" in row:
        s += f" | KL {row['kl']:6.3f}"
    return s + f" | matcher {row['matcher']:6.1%}"


def disjoint(a, b):
    """Do two [lo, hi] intervals fail to overlap? The only evidence a cell is DECIDED at n=48."""
    return a[1] < b[0] or b[1] < a[0]

In [ ]:
# %% 1.2 Use the re-fitted thresholds if notebook 09 has produced them.
#
# The detector was fitted on `no-steer` against `dense/all m=2`, both layer 16, and gen-only's
# apology loops clear all three thresholds while sitting just under each (WORKLOG 20). A verdict
# about gen-only computed on those thresholds inherits that slack, so if better ones exist, use
# them -- and report both rates regardless, because the gap between them IS the instrument risk.
FIT_ANCHOR = dict(FIT)
REFIT_SOURCE = "week3 anchors (no-steer vs dense/all m=2)"
try:
    _lab = json.load(open(adass.artifact("week6_labels.json")))
    _new = _lab.get("new_thresholds")
    if _new:
        FIT = {k: dict(FIT_ANCHOR[k], threshold=v) for k, v in _new.items()}
        REFIT_SOURCE = "week6 hand labels (notebook 09)"
except FileNotFoundError:
    pass
print(f"coherence thresholds in use: {REFIT_SOURCE}")
assert not REFIT_SOURCE.startswith("week3"), (
    "This notebook REQUIRES the re-fitted thresholds -- H1's verdict is exactly the thing that\n"
    "depends on which ruler is used. Run notebooks/09 first, or git pull so\n"
    "data/results/week6_labels.json is present.")
if False:
    print()
    print("!" * 78)
    print("!! FALLING BACK TO THE ANCHOR THRESHOLDS. 108 hand labels showed these agree with")
    print("!! a human 31% of the time on gen-only -- the arm the verdict below turns on. This")
    print("!! run would produce a verdict computed with a ruler known to be wrong for one arm.")
    print("!! Fix: git pull so data/results/week6_labels.json is present, then re-run.")
    print("!" * 78)
    print()
for k in FIT:
    print(f"   {k:12s} {FIT_ANCHOR[k]['threshold']:8.3f} -> {FIT[k]['threshold']:8.3f}")


def mech_broken_with(texts, fit):
    return [adass.classify_mechanical(t, fit)["broken"] for t in texts]

In [ ]:
# %% 1.2 The replication gate, plus the negative control and the KL reference text.
#
# Three things at once, all from the unsteered and the dense rel-1.0 conditions:
#   - REF_TEXTS  -- the fixed unsteered continuation every KL in this notebook is measured on;
#   - the negative control -- unsteered must be ~0% suppressed and ~0% broken, or the
#     instruments are wrong before any comparison starts;
#   - the gate -- dense at rel 1.0 must land where week 4 §7 left it.
if LOAD_MODEL:
    HN10 = adass.mean_hidden_norm(MODEL, TOK, TO_CHAT, PROMPTS, LAYER, device=DEV)
    print(f"mean ||h|| at layer {LAYER}: {HN10:.1f}  (week 4 measured 170.9)")

    REF_TEXTS = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0,
                               max_new_tokens=KL_REF_TOKENS, batch_size=GEN_BS,
                               device=DEV, dtype=DT)
    ns_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, mult=0.0, max_new_tokens=MAXNEW,
                             batch_size=GEN_BS, device=DEV, dtype=DT)
    ns = score(ns_gens, PROMPTS)
    print(fmt(ns, "no-steer (negative control)"))

    V_ref = adass.rel_norm_rows(V, R16 * 1.0, HN10)      # the operating point, norm-matched
    d10_gens = adass.generate(MODEL, TOK, TO_CHAT, PROMPTS, layer=LAYER, vector=V_ref, mult=1.0,
                              positions="all", max_new_tokens=MAXNEW, batch_size=GEN_BS,
                              device=DEV, dtype=DT)
    d10 = score(d10_gens, PROMPTS)
    d10.update(adass.strength_row(V_ref, 1.0, HN10), label="dense", rel_factor=1.0,
               method="dense", sparsity=0.0, positions="all", n_prompts=len(PROMPTS))
    print(fmt(d10, "dense/all rel 1.0"))

    # Score the gate row under the thresholds the stored reference was produced with, so the
    # comparison measures the pipeline rather than the recalibration. See the note above §1.2.
    _ab = mech_broken_with(d10_gens, FIT_ANCHOR)
    _anchor_broken = sum(_ab) / len(_ab)
    _anchor_clean = sum((not b) and (not a)
                        for b, a in zip(_ab, d10["judge_answered"])) / len(_ab)
    print(f"dense/all rel 1.0 under the ANCHOR thresholds: broken {_anchor_broken:.1%} "
          f"clean {_anchor_clean:.1%}   (the gate compares these)")
    w4 = json.load(open(adass.artifact("week4_layers.json")))["s7_relative_grid"]["cells"]["L10/rel1.0"]["row"]
    print(f"\nweek 4 §7 L10/rel1.0: broken {w4['broken']:.1%}  clean {w4['clean_refusal']:.1%}")

    # The bar is 10 points, not equality: the 24 August run reproduced every rate in the sweep to
    # within 4.2% while only 14-48% of individual generations matched token-for-token, because a
    # T4 has no bfloat16 and greedy decoding is deterministic given identical numerics and not
    # otherwise. Rates are dtype-invariant here; text is not.
    gate = dict(
        neg_control_ok=bool(ns["broken"] <= 0.05 and ns["suppressed"] <= 0.05),
        clean_delta=_anchor_clean - w4["clean_refusal"],
        broken_delta=_anchor_broken - w4["broken"],
        h_norm=HN10, h_norm_w4=CONFIG["h_norms"][str(LAYER)])
    gate["repro_ok"] = bool(abs(gate["clean_delta"]) <= 0.10 and abs(gate["broken_delta"]) <= 0.10)
    gate["pass_"] = bool(gate["neg_control_ok"] and gate["repro_ok"])
    print(f"\nnegative control {'PASS' if gate['neg_control_ok'] else 'FAIL'} | "
          f"replication delta clean {gate['clean_delta']:+.1%} broken {gate['broken_delta']:+.1%} "
          f"-> {'PASS' if gate['repro_ok'] else 'FAIL'}")
    RESULTS["s1_gate"] = dict(gate=gate, no_steer=ns, dense_rel1=d10)
    RESULTS["s1_ref_texts"] = REF_TEXTS
    print(adass.save_results(RESULTS, OUT))
    assert gate["pass_"], "BLOCKING: fix this before running anything below."
else:
    HN10 = CONFIG["h_norms"][str(LAYER)]
    REF_TEXTS = (PRIOR.get("s1_ref_texts") or None)
    ns = d10 = None
    print(f"deferred: needs ADASS_LOAD_MODEL=1. Using stored ||h|| = {HN10}")

In [ ]:
# %% 1.3 The prompts nothing has touched, and the masks built on them.
REL_PRIMARY, REL_SECONDARY = 2.0, 1.5

if LOAD_MODEL:
    P192 = adass.make_splits(seed=CONFIG["seed"], test_n=192)["harmless_test"]
    assert P192[:48] == PROMPTS, "the extended split moved the original 48 -- STOP"
    assert P192[:144] == adass.make_splits(seed=CONFIG["seed"], test_n=144)["harmless_test"], \
        "the extended split moved the prompts notebooks 08 and 10 used -- STOP"
    FRESH = P192[144:]
    assert len(FRESH) == 48
    print(f"{len(FRESH)} prompts. Used by no run, no mask, no threshold fit and no selection step.")
    print("first two:", *[f"  - {p}" for p in FRESH[:2]], sep="\n")

    MU = adass.last_token_hidden(MODEL, TOK, TO_CHAT, SPL["harmless_train"],
                                 device=DEV).mean(1)[LAYER + 1]
    H_F = adass.last_token_hidden(MODEL, TOK, TO_CHAT, FRESH, device=DEV)[LAYER + 1]
    print("mu and per-prompt hidden states ready")

    def build(method, sparsity=0.90):
        if method is None:
            return V
        if method == "static":
            return (V * adass.static_mask(V, sparsity)).unsqueeze(0).expand(len(FRESH), -1).contiguous()
        out = torch.empty(len(FRESH), V.numel())
        for i in range(len(FRESH)):
            m = (adass.adaptive_absproj_mask(V, H_F[i], MU, sparsity) if method == "absproj"
                 else adass.adaptive_signed_mask(V, H_F[i], MU, sparsity))
            out[i] = V * m
        return out


def run_arm(label, base, rel):
    vec = adass.rel_norm_rows(base, R16 * rel, HN10)
    g = adass.generate(MODEL, TOK, TO_CHAT, FRESH, layer=LAYER, vector=vec, mult=1.0,
                       positions="all", max_new_tokens=MAXNEW, batch_size=GEN_BS,
                       device=DEV, dtype=DT)
    row = score(g, FRESH)
    row.update(adass.strength_row(vec, 1.0, HN10), label=label, rel_factor=rel,
               n_prompts=len(FRESH))
    ob = mech_broken_with(g, FIT_ANCHOR)
    row["broken_anchor"] = sum(ob) / len(ob)
    adass.empty_cache(DEV)
    return row, g

## §2 The rule, fixed before the run

> **H1, damage half.** At sparsity 0.90 and matched relative strength, per-input `absproj` masking
> **beats** a comparator when their Wilson intervals on `broken` are disjoint with absproj lower,
> **and** their intervals on `suppressed` overlap — the effect has to be matched for a damage
> comparison to mean anything.
>
> - beats **both** dense and static at `REL_PRIMARY` → **H1-DAMAGE SUPPORTED**
> - beats **dense only** → **PARTIAL**: sparsification helps, but no more than the cheap static mask
>   does, and the per-input half of H1 is what is unsupported
> - beats **neither** → **NOT SUPPORTED**, and correction 23's re-scored reversal was an artefact of
>   scoring prompts the thresholds had been fitted near
>
> **Primary strength is `REL_PRIMARY` = 2.0**, for continuity with corrections 19 and 22, which is
> where the contested numbers live. **`REL_SECONDARY` = 1.5 is declared here as a secondary
> comparison**, not as a second chance: under the corrected thresholds dense is *saturated* at 100%
> broken at rel 2.0, so a win there could be an artefact of the comparator having nowhere to go. At
> rel 1.5 dense sits near 69% and has room. If the two strengths disagree, the verdict is the primary
> one and the disagreement is reported as the finding.

**Why `signed` is in the table and not in the rule.** It is the third adaptive scheme and it has lost
every comparison it has been in. Running it costs two minutes and keeps the table honest about how
much of H1 rides on one scheme; nothing in the verdict depends on it.

**What a win would mean.** That choosing *which dimensions to steer per input* halves the damage at
matched effect — the AdaSS thesis in its original form, on prompts chosen before the method was, with
an instrument fixed before the run and validated against human labels. That is the strongest form
this project can put the claim in.

In [ ]:
# %% 2.1 Three arms plus the diagnostic, at both strengths.
ARMS = [("dense", None), ("static-0.90", "static"),
        ("absproj-0.90", "absproj"), ("signed-0.90", "signed")]

if LOAD_MODEL:
    h1 = {}
    for rel in (REL_PRIMARY, REL_SECONDARY):
        for label, method in ARMS:
            row, gens = run_arm(label, build(method), rel)
            h1[f"{label}/rel{rel}"] = dict(row=row, gens=gens)
            print(f"{fmt(row, f'{label:14s} rel x{rel}')} | broken@anchor {row['broken_anchor']:5.1%}")
    RESULTS["s2_h1"] = dict(cells=h1, rel_primary=REL_PRIMARY, rel_secondary=REL_SECONDARY,
                            n=len(FRESH), prompts_from="harmless_test[144:192]",
                            thresholds=REFIT_SOURCE)
    print(adass.save_results(RESULTS, OUT))
else:
    print("deferred: needs ADASS_LOAD_MODEL=1")

In [ ]:
# %% 2.2 The registered verdict.
s2 = stored("s2_h1")
if s2:
    R = {k: v["row"] for k, v in s2["cells"].items()}
    for rel in (s2["rel_primary"], s2["rel_secondary"]):
        print(f"-- relative strength x{rel}")
        print(f"   {'arm':14s} {'broken':>8s} {'interval':>14s} {'suppressed':>11s} {'clean':>8s}")
        for label, _ in [("dense", 0), ("static-0.90", 0), ("absproj-0.90", 0), ("signed-0.90", 0)]:
            r = R.get(f"{label}/rel{rel}")
            if not r:
                continue
            ci = f"[{r['broken_ci'][0]:.2f},{r['broken_ci'][1]:.2f}]"
            print(f"   {label:14s} {r['broken']:8.1%} {ci:>14s} {r['suppressed']:11.1%} "
                  f"{r['clean_refusal']:8.1%}")
        print()

    def beats(rel, other):
        a, b = R[f"absproj-0.90/rel{rel}"], R[f"{other}/rel{rel}"]
        matched = not disjoint(a["suppressed_ci"], b["suppressed_ci"])
        lower = disjoint(a["broken_ci"], b["broken_ci"]) and a["broken"] < b["broken"]
        print(f"   absproj vs {other:12s} rel x{rel}: effect matched={matched} "
              f"less damage (disjoint)={lower} -> {'BEATS' if (matched and lower) else 'not decided'}")
        return bool(matched and lower)

    print("PRIMARY:")
    p_dense = beats(s2["rel_primary"], "dense")
    p_static = beats(s2["rel_primary"], "static-0.90")
    print("SECONDARY (declared in advance; does not decide):")
    s_dense = beats(s2["rel_secondary"], "dense")
    s_static = beats(s2["rel_secondary"], "static-0.90")

    verdict = ("H1-DAMAGE SUPPORTED" if (p_dense and p_static) else
               "PARTIAL -- beats dense, not static" if p_dense else
               "NOT SUPPORTED on unseen prompts")
    agree = (p_dense, p_static) == (s_dense, s_static)
    print(f"\nVERDICT (primary, rel x{s2['rel_primary']}): {verdict}")
    print(f"secondary agrees with primary: {agree}"
          + ("" if agree else "  -- REPORT THE DISAGREEMENT; a win only at the strength where the"
                              " comparator is saturated is not a win"))
    RESULTS["s2_verdict"] = dict(verdict=verdict, primary_beats_dense=p_dense,
                                 primary_beats_static=p_static, secondary_beats_dense=s_dense,
                                 secondary_beats_static=s_static, secondary_agrees=bool(agree),
                                 thresholds=s2["thresholds"], n=s2["n"],
                                 axis="broken, at matched suppression (registered in §2)")
    print(adass.save_results(RESULTS, OUT))

    print("\nRead four (HANDOVER trap 8):")
    for k in (f"absproj-0.90/rel{s2['rel_primary']}", f"static-0.90/rel{s2['rel_primary']}"):
        for i in (0, 1):
            print(f"\n--- {k}")
            print(s2["cells"][k]["gens"][i][:300])
else:
    print("deferred: §2.1 has not run")

## §3 What this settles

In [ ]:
# %% 3.1 Summary, and the comparison against every earlier reading of H1.
v = stored("s2_verdict")
print("=" * 74)
print(f"{'registered H1 damage test':30s} {v['verdict'] if v else 'not run'}")
if v:
    print(f"{'thresholds':30s} {v['thresholds']}")
    print(f"{'secondary agrees':30s} {v['secondary_agrees']}")
print("=" * 74)
print("""
How H1 has read, in order:
  wk2  matched multiplier, no controls          adaptive no better than static
  wk3  matched KL, graded margin                never worse, wins 5 of 12
  c19  matched relative strength, n=48/96       undecided on clean refusal
  c22  damage axis, 48 unseen, anchor ruler     NOT SUPPORTED
  c23  same generations, corrected ruler        reverses -- recorded, not claimed
  here registered, corrected ruler, 48 unseen   the line above this one
""")
RESULTS["meta"] = dict(notebook="11_h1_registered", layer=LAYER,
                       prompts="harmless_test[144:192]",
                       registered="§2's rule and both strengths were fixed above the code before "
                                  "the run; the thresholds were frozen and published beforehand")
print(adass.save_results(RESULTS, OUT))